# LeetCode 399: Evaluate Division

**Difficulty**: Medium  
**Topics**: Graph, DFS, BFS, Union Find, Weighted Graph  
**Link**: [LeetCode Problem](https://leetcode.com/problems/evaluate-division/)

---

## Understanding the Problem

### Key Concepts

1. **Division as Graph**: Each equation `a / b = value` creates a directed edge
2. **Weighted Edges**: Edge weight represents the division result
3. **Bidirectional**: If `a / b = 2.0`, then `b / a = 0.5`
4. **Path Multiplication**: To find `a / c`, multiply weights along path a → b → c

### Visual Example

```
Given:
  a / b = 2.0  →  a --2.0--> b
  b / c = 3.0  →  b --3.0--> c

Graph:
  a --2.0--> b --3.0--> c
  a <--0.5-- b <--0.33-- c

Query: a / c = ?
Path: a → b → c
Answer: 2.0 × 3.0 = 6.0

Query: c / a = ?
Path: c → b → a
Answer: 0.33 × 0.5 = 0.166... (or 1/6)
```

### Problem Insight

This is a **weighted graph path-finding** problem:
- Variables are nodes
- Divisions are weighted directed edges
- Query answer = product of edge weights along path
- If no path exists, return -1.0

## Approach 1: DFS (Depth-First Search)

### Key Idea

Build a weighted directed graph and use DFS to find paths between query variables.

### Algorithm

1. **Build Graph**:
   - For each equation `a / b = value`:
     - Add edge: `a → b` with weight `value`
     - Add edge: `b → a` with weight `1/value`

2. **For Each Query** `(src, dst)`:
   - If either variable not in graph: return -1.0
   - If `src == dst`: return 1.0
   - Use DFS to find path from src to dst
   - Multiply edge weights along the path

3. **DFS Process**:
   - Mark current node as visited
   - For each neighbor:
     - If neighbor is destination: return edge weight
     - If not visited: recursively DFS
     - If path found: return current_weight × recursive_result

### Complexity

- **Time**: O(Q × (V + E)) where Q = queries, V = variables, E = equations
- **Space**: O(V + E) for graph + O(V) for recursion stack

### Why This Works

DFS explores all possible paths, multiplying weights to get division result.

In [ ]:
from collections import defaultdict

def calcEquation_dfs(equations, values, queries):
    """
    DFS approach to evaluate division.
    Time: O(Q × (V + E))
    Space: O(V + E)
    """
    # Build weighted directed graph
    graph = defaultdict(dict)
    
    for (dividend, divisor), value in zip(equations, values):
        graph[dividend][divisor] = value
        graph[divisor][dividend] = 1.0 / value
    
    def dfs(src, dst, visited):
        """Find path from src to dst and return product of weights."""
        # Variable not in graph
        if src not in graph or dst not in graph:
            return -1.0
        
        # Same variable
        if src == dst:
            return 1.0
        
        # Mark as visited
        visited.add(src)
        
        # Explore neighbors
        for neighbor, weight in graph[src].items():
            if neighbor not in visited:
                # Recursively find path from neighbor to dst
                result = dfs(neighbor, dst, visited)
                
                # If path found, multiply weights
                if result != -1.0:
                    return weight * result
        
        # No path found
        return -1.0
    
    # Process each query
    results = []
    for dividend, divisor in queries:
        result = dfs(dividend, divisor, set())
        results.append(result)
    
    return results

# Test
print("Approach 1: DFS\n")
equations = [["a","b"],["b","c"]]
values = [2.0, 3.0]
queries = [["a","c"],["b","a"],["a","e"],["a","a"],["x","x"]]

print(f"Equations: {equations}")
print(f"Values: {values}")
print(f"Queries: {queries}")
print(f"Results: {calcEquation_dfs(equations, values, queries)}")
print(f"Expected: [6.0, 0.5, -1.0, 1.0, -1.0]")

## Detailed DFS Trace

In [ ]:
def calcEquation_dfs_verbose(equations, values, queries):
    """
    DFS with detailed trace.
    """
    # Build graph
    graph = defaultdict(dict)
    
    print("Building Graph:\n")
    for (dividend, divisor), value in zip(equations, values):
        print(f"  {dividend} / {divisor} = {value}")
        print(f"    → Add edge: {dividend} --{value}--> {divisor}")
        print(f"    → Add edge: {divisor} --{1.0/value:.4f}--> {dividend}\n")
        graph[dividend][divisor] = value
        graph[divisor][dividend] = 1.0 / value
    
    print("Graph Structure:")
    for node, neighbors in sorted(graph.items()):
        print(f"  {node}: {dict(neighbors)}")
    print("\n" + "="*80 + "\n")
    
    def dfs(src, dst, visited, depth=0):
        """DFS with trace."""
        indent = "  " * depth
        
        print(f"{indent}DFS({src} → {dst})")
        
        # Check if variables exist
        if src not in graph:
            print(f"{indent}  {src} not in graph → return -1.0")
            return -1.0
        if dst not in graph:
            print(f"{indent}  {dst} not in graph → return -1.0")
            return -1.0
        
        # Same variable
        if src == dst:
            print(f"{indent}  {src} == {dst} → return 1.0")
            return 1.0
        
        # Mark visited
        visited.add(src)
        print(f"{indent}  Visited: {sorted(visited)}")
        print(f"{indent}  Neighbors of {src}: {list(graph[src].keys())}")
        
        # Explore neighbors
        for neighbor, weight in graph[src].items():
            if neighbor not in visited:
                print(f"{indent}  → Exploring {src} --{weight}--> {neighbor}")
                result = dfs(neighbor, dst, visited, depth + 1)
                
                if result != -1.0:
                    final = weight * result
                    print(f"{indent}  ← Path found! {weight} × {result} = {final}")
                    return final
                else:
                    print(f"{indent}  ← No path through {neighbor}")
        
        print(f"{indent}  No path from {src} to {dst} → return -1.0")
        return -1.0
    
    # Process queries
    results = []
    for idx, (dividend, divisor) in enumerate(queries, 1):
        print(f"Query {idx}: {dividend} / {divisor} = ?\n")
        print("-" * 80)
        result = dfs(dividend, divisor, set())
        results.append(result)
        print(f"\nResult: {result}")
        print("="*80 + "\n")
    
    return results

# Trace
print("DFS Detailed Trace\n")
equations = [["a","b"],["b","c"]]
values = [2.0, 3.0]
queries = [["a","c"],["b","a"]]

calcEquation_dfs_verbose(equations, values, queries)

## Approach 2: BFS (Breadth-First Search)

### Key Idea

Use BFS to find the shortest path and multiply weights along the way.

### Algorithm

1. Build the same weighted directed graph
2. For each query, use BFS with a queue
3. Store (node, accumulated_product) in queue
4. When destination found, return accumulated product

### Complexity

- **Time**: O(Q × (V + E))
- **Space**: O(V + E) for graph + O(V) for queue

### Advantage

BFS finds shortest path (fewest edges), though any path gives correct answer.

In [ ]:
from collections import deque

def calcEquation_bfs(equations, values, queries):
    """
    BFS approach to evaluate division.
    Time: O(Q × (V + E))
    Space: O(V + E)
    """
    # Build graph
    graph = defaultdict(dict)
    
    for (dividend, divisor), value in zip(equations, values):
        graph[dividend][divisor] = value
        graph[divisor][dividend] = 1.0 / value
    
    def bfs(src, dst):
        """Find path using BFS."""
        # Check if variables exist
        if src not in graph or dst not in graph:
            return -1.0
        
        # Same variable
        if src == dst:
            return 1.0
        
        # BFS with queue: (node, accumulated_product)
        queue = deque([(src, 1.0)])
        visited = {src}
        
        while queue:
            node, product = queue.popleft()
            
            # Check all neighbors
            for neighbor, weight in graph[node].items():
                if neighbor == dst:
                    # Found destination
                    return product * weight
                
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, product * weight))
        
        # No path found
        return -1.0
    
    # Process queries
    return [bfs(dividend, divisor) for dividend, divisor in queries]

# Test
print("Approach 2: BFS\n")
equations = [["a","b"],["b","c"]]
values = [2.0, 3.0]
queries = [["a","c"],["b","a"],["a","e"],["a","a"],["x","x"]]

print(f"Equations: {equations}")
print(f"Values: {values}")
print(f"Queries: {queries}")
print(f"Results: {calcEquation_bfs(equations, values, queries)}")
print(f"Expected: [6.0, 0.5, -1.0, 1.0, -1.0]")

## Approach 3: Union-Find with Weights

### Key Idea

Use Union-Find to group variables in same component, storing division ratios.

### Algorithm

1. Each variable starts as its own parent with weight 1.0
2. For each equation `a / b = value`:
   - Union a and b
   - Store weight relationship
3. For each query:
   - If same component, calculate ratio using stored weights
   - Otherwise, return -1.0

### Complexity

- **Time**: O((E + Q) × α(V)) ≈ O(E + Q)
- **Space**: O(V)

### Advantage

Efficient for multiple queries on same dataset.

In [ ]:
def calcEquation_unionfind(equations, values, queries):
    """
    Union-Find with weights.
    Time: O((E + Q) × α(V))
    Space: O(V)
    """
    parent = {}
    weight = {}  # weight[x] = x / parent[x]
    
    def find(x):
        """Find root with path compression and weight update."""
        if x not in parent:
            parent[x] = x
            weight[x] = 1.0
            return x
        
        if parent[x] != x:
            # Path compression with weight update
            original_parent = parent[x]
            root = find(parent[x])
            parent[x] = root
            weight[x] *= weight[original_parent]
        
        return parent[x]
    
    def union(x, y, value):
        """Union x and y where x / y = value."""
        root_x = find(x)
        root_y = find(y)
        
        if root_x != root_y:
            # Make root_y the parent of root_x
            parent[root_x] = root_y
            # weight[root_x] = root_x / root_y
            # We know: x / y = value
            # We have: x / root_x = weight[x], y / root_y = weight[y]
            # So: root_x / root_y = (x / root_x) / (y / root_y) / (x / y)
            weight[root_x] = weight[y] * value / weight[x]
    
    def query(x, y):
        """Calculate x / y."""
        if x not in parent or y not in parent:
            return -1.0
        
        root_x = find(x)
        root_y = find(y)
        
        if root_x != root_y:
            return -1.0
        
        # x / y = (x / root) / (y / root) = weight[x] / weight[y]
        return weight[x] / weight[y]
    
    # Build union-find structure
    for (dividend, divisor), value in zip(equations, values):
        union(dividend, divisor, value)
    
    # Process queries
    return [query(dividend, divisor) for dividend, divisor in queries]

# Test
print("Approach 3: Union-Find\n")
equations = [["a","b"],["b","c"]]
values = [2.0, 3.0]
queries = [["a","c"],["b","a"],["a","e"],["a","a"],["x","x"]]

print(f"Equations: {equations}")
print(f"Values: {values}")
print(f"Queries: {queries}")
print(f"Results: {calcEquation_unionfind(equations, values, queries)}")
print(f"Expected: [6.0, 0.5, -1.0, 1.0, -1.0]")

## Edge Cases and Special Scenarios

In [ ]:
print("Edge Cases Testing\n")
print("="*70)

# Case 1: Same variable
print("\nCase 1: Query same variable")
equations = [["a","b"]]
values = [2.0]
queries = [["a","a"], ["b","b"]]
print(f"Equations: {equations}, Values: {values}")
print(f"Queries: {queries}")
print(f"DFS: {calcEquation_dfs(equations, values, queries)}")
print(f"BFS: {calcEquation_bfs(equations, values, queries)}")
print(f"Union-Find: {calcEquation_unionfind(equations, values, queries)}")
print("Explanation: Any variable divided by itself = 1.0\n")

# Case 2: Variable not in graph
print("Case 2: Variable not in graph")
equations = [["a","b"]]
values = [2.0]
queries = [["a","c"], ["x","y"]]
print(f"Equations: {equations}, Values: {values}")
print(f"Queries: {queries}")
print(f"DFS: {calcEquation_dfs(equations, values, queries)}")
print(f"BFS: {calcEquation_bfs(equations, values, queries)}")
print(f"Union-Find: {calcEquation_unionfind(equations, values, queries)}")
print("Explanation: Unknown variables return -1.0\n")

# Case 3: Disconnected components
print("Case 3: Disconnected components")
equations = [["a","b"],["c","d"]]
values = [2.0, 3.0]
queries = [["a","d"], ["b","c"]]
print(f"Equations: {equations}, Values: {values}")
print(f"Queries: {queries}")
print(f"DFS: {calcEquation_dfs(equations, values, queries)}")
print(f"BFS: {calcEquation_bfs(equations, values, queries)}")
print(f"Union-Find: {calcEquation_unionfind(equations, values, queries)}")
print("Explanation: No path between components = -1.0\n")

# Case 4: Long chain
print("Case 4: Long chain")
equations = [["a","b"],["b","c"],["c","d"],["d","e"]]
values = [2.0, 3.0, 4.0, 5.0]
queries = [["a","e"], ["e","a"]]
print(f"Equations: {equations}, Values: {values}")
print(f"Queries: {queries}")
print(f"DFS: {calcEquation_dfs(equations, values, queries)}")
print(f"BFS: {calcEquation_bfs(equations, values, queries)}")
print(f"Union-Find: {calcEquation_unionfind(equations, values, queries)}")
print(f"Explanation: a/e = 2×3×4×5 = 120, e/a = 1/120\n")

# Case 5: Cycle in graph
print("Case 5: Cycle in graph")
equations = [["a","b"],["b","c"],["c","a"]]
values = [2.0, 3.0, 1.0/6.0]
queries = [["a","c"], ["b","a"]]
print(f"Equations: {equations}, Values: {values}")
print(f"Queries: {queries}")
print(f"DFS: {calcEquation_dfs(equations, values, queries)}")
print(f"BFS: {calcEquation_bfs(equations, values, queries)}")
print(f"Union-Find: {calcEquation_unionfind(equations, values, queries)}")
print("Explanation: Cycle is valid, any path works")

## Visual Graph Representation

In [ ]:
def visualize_graph(equations, values):
    """
    Visualize the weighted directed graph.
    """
    graph = defaultdict(dict)
    
    print("Building Graph from Equations:\n")
    for (dividend, divisor), value in zip(equations, values):
        graph[dividend][divisor] = value
        graph[divisor][dividend] = 1.0 / value
        print(f"  {dividend} / {divisor} = {value}")
    
    print("\nGraph Structure (Adjacency List):\n")
    for node in sorted(graph.keys()):
        print(f"  {node}:")
        for neighbor, weight in sorted(graph[node].items()):
            print(f"    → {neighbor} (weight: {weight:.4f})")
    
    print("\nEdges (Directed):\n")
    edges_shown = set()
    for node in sorted(graph.keys()):
        for neighbor, weight in sorted(graph[node].items()):
            if (node, neighbor) not in edges_shown:
                reverse_weight = graph[neighbor][node]
                print(f"  {node} --{weight:.4f}--> {neighbor}")
                print(f"  {neighbor} --{reverse_weight:.4f}--> {node}")
                edges_shown.add((node, neighbor))
                edges_shown.add((neighbor, node))
                print()

# Visualize
print("Graph Visualization\n")
print("="*70)

print("\nExample 1:")
visualize_graph([["a","b"],["b","c"]], [2.0, 3.0])

print("\n" + "="*70)
print("\nExample 2:")
visualize_graph([["a","b"],["b","c"],["c","d"]], [2.0, 3.0, 4.0])

## Comparison of Approaches

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **DFS** | O(Q × (V + E)) | O(V + E) | Simple, intuitive | May explore unnecessary paths |
| **BFS** | O(Q × (V + E)) | O(V + E) | Finds shortest path | Requires queue |
| **Union-Find** | O((E + Q) × α(V)) | O(V) | Efficient for many queries | Complex implementation |

Where:
- Q = number of queries
- V = number of variables
- E = number of equations
- α(V) = inverse Ackermann (nearly constant)

### When to Use Each

- **DFS**: Simple implementation, recursive thinking
- **BFS**: Prefer iterative, shortest path
- **Union-Find**: Many queries on same dataset, dynamic connectivity

## Key Takeaways

### Problem Type

**Weighted Graph Path Finding**: Find path in weighted directed graph and multiply edge weights.

### Core Insight

Division creates a weighted directed graph:
- Variables are nodes
- `a / b = value` creates edge `a → b` with weight `value`
- Reverse edge: `b → a` with weight `1/value`
- Query answer = product of weights along path

### DFS Algorithm

```python
def dfs(src, dst, visited):
    if src not in graph or dst not in graph:
        return -1.0
    if src == dst:
        return 1.0
    
    visited.add(src)
    for neighbor, weight in graph[src].items():
        if neighbor not in visited:
            result = dfs(neighbor, dst, visited)
            if result != -1.0:
                return weight * result
    return -1.0
```

### BFS Algorithm

```python
def bfs(src, dst):
    if src not in graph or dst not in graph:
        return -1.0
    if src == dst:
        return 1.0
    
    queue = deque([(src, 1.0)])
    visited = {src}
    
    while queue:
        node, product = queue.popleft()
        for neighbor, weight in graph[node].items():
            if neighbor == dst:
                return product * weight
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, product * weight))
    return -1.0
```

### Key Insights

1. **Bidirectional edges**: If `a/b = x`, then `b/a = 1/x`
2. **Path multiplication**: Multiply weights along path
3. **Same variable**: `a/a = 1.0` always
4. **No path**: Return -1.0 if variables disconnected
5. **Mark visited**: Avoid infinite loops in cycles

### Common Mistakes

❌ **Forgetting reverse edges**
```python
# WRONG - only one direction
graph[a][b] = value
```

✅ **Add both directions**
```python
# CORRECT
graph[a][b] = value
graph[b][a] = 1.0 / value
```

❌ **Not checking if variables exist**
```python
# WRONG - may cause KeyError
if src == dst:
    return 1.0
```

✅ **Check existence first**
```python
# CORRECT
if src not in graph or dst not in graph:
    return -1.0
if src == dst:
    return 1.0
```

❌ **Not marking visited**
```python
# WRONG - infinite loop in cycles
for neighbor, weight in graph[src].items():
    result = dfs(neighbor, dst, visited)
```

✅ **Always mark visited**
```python
# CORRECT
visited.add(src)
for neighbor, weight in graph[src].items():
    if neighbor not in visited:
        result = dfs(neighbor, dst, visited)
```

### Edge Cases to Remember

1. **Same variable**: `a/a = 1.0`
2. **Unknown variable**: Return -1.0
3. **Disconnected components**: No path = -1.0
4. **Long chains**: Multiply all weights
5. **Cycles**: Valid, any path works

### Complexity Analysis

**DFS/BFS:**
- **Time**: O(Q × (V + E))
  - Q queries
  - Each query: O(V + E) to explore graph
- **Space**: O(V + E)
  - Graph storage
  - Visited set/recursion stack

**Union-Find:**
- **Time**: O((E + Q) × α(V)) ≈ O(E + Q)
  - Build: O(E × α(V))
  - Query: O(Q × α(V))
- **Space**: O(V)
  - Parent and weight arrays

### Related Problems

- **Path with Maximum Probability** (LeetCode 1514)
- **Network Delay Time** (LeetCode 743)
- **Cheapest Flights Within K Stops** (LeetCode 787)
- **Accounts Merge** (LeetCode 721)

### Remember

🎯 **Division = weighted directed graph**  
🎯 **Bidirectional edges** with reciprocal weights  
🎯 **Path product** = query answer  
🎯 **Mark visited** to avoid cycles  
🎯 **Check existence** before processing  
🎯 **DFS/BFS** for path finding  
🎯 **Union-Find** for many queries  

Master this problem and you'll understand:
- Weighted directed graphs
- DFS and BFS on weighted graphs
- Union-Find with weights
- Path multiplication in graphs

## Problem Statement

You are given an array of variable pairs `equations` and an array of real numbers `values`, where `equations[i] = [Ai, Bi]` and `values[i]` represent the equation `Ai / Bi = values[i]`. Each `Ai` or `Bi` is a string that represents a single variable.

You are also given some `queries`, where `queries[j] = [Cj, Dj]` represents the jth query where you must find the answer for `Cj / Dj = ?`.

Return the answers to all queries. If a single answer cannot be determined, return `-1.0`.

**Note**: The input is always valid. You may assume that evaluating the queries will not result in division by zero and that there is no contradiction.

### Examples

**Example 1:**
```
Input: 
  equations = [["a","b"],["b","c"]]
  values = [2.0,3.0]
  queries = [["a","c"],["b","a"],["a","e"],["a","a"],["x","x"]]
  
Output: [6.00000,0.50000,-1.00000,1.00000,-1.00000]

Explanation:
  Given: a / b = 2.0, b / c = 3.0
  queries are: a / c = ?, b / a = ?, a / e = ?, a / a = ?, x / x = ?
  return: [6.0, 0.5, -1.0, 1.0, -1.0 ]
```

**Example 2:**
```
Input: 
  equations = [["a","b"],["b","c"],["bc","cd"]]
  values = [1.5,2.5,5.0]
  queries = [["a","c"],["c","b"],["bc","cd"],["cd","bc"]]
  
Output: [3.75000,0.40000,5.00000,0.20000]
```

### Constraints

- `1 <= equations.length <= 20`
- `equations[i].length == 2`
- `1 <= Ai.length, Bi.length <= 5`
- `values.length == equations.length`
- `0.0 < values[i] <= 20.0`
- `1 <= queries.length <= 20`
- `queries[i].length == 2`
- `1 <= Cj.length, Dj.length <= 5`
- `Ai, Bi, Cj, Dj` consist of lower case English letters and digits.